# Question 1 — Scaling Transformation

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_image(img, title="Image", cmap=None):
    plt.figure(figsize=(6, 6))
    if cmap:
        plt.imshow(img, cmap=cmap)
    else:
        # Convert BGR to RGB for correct color display in Matplotlib
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

image_path = 'tel_aviv_is_red.jpg'
img = cv2.imread(image_path)
show_image(img, "Original Image")

## 1. Definition and Geometric Meaning

Scaling transformation is a transformation that changes the size of a 2D image and enlarges or shrinks it.
It does so by multiplying the coordinates of the original image by a scaling factor.
If the scaling factor is 1 the image remains the same.

If the scaling factor is larger 1 the objects become bigger, the distance between two points in the image is increased proportionally to the scaling factor and the shape and proportions are affected according to the scaling factor for each axes - if it is the same the proportions remains the same ,otherwise the image stretches or squashed.

If the scaling factor is smaller 1 the objects become smaller, the distance between two points in the image is decreased proportionally to the scaling factor and the shape and proportions are affected according to the scaling factor for each axes - if it is the same the proportions remains the same ,otherwise the image stretches or squashed.

## 2. Scaling Matrix Derivation

Scaling matrix format

$$S = \begin{bmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

$s_x$: Scaling factor for the x axis (horizontal stretching or shrinking)
$s_y$: Scaling factor for the y axis (vertical stretching or shrinking)
$0$: The off diagonal zeros ensure that during the scaling the x values doesn't affect the new y values.
$1$: The bottom-right 1 is the extra dimension added to preserves the homogeneous coordinate so that the point can be used in further transformations that require addition such as translation.

Homogeneous coordinates are used to enable transformations that apart from multiplication also include addition such as translation. A 2x2 transformation matrix is not able to perform addition so we add another dimension with the value 1 and bu doing so addition to the original values can be achieved in matrix multiplication.

## 3. Uniform vs. Non-Uniform Scaling

If $s_x$ = $s_y$ scaling along both axes is the same and the scaling is uniform.
Aspect ratio of the original image is remained, the shape still looks like the original but bigger or smaller and geometric properties remain the same.

If not, the scaling along both axes is not the same and the scaling is non-uniform.
Aspect ratio is not preserved and changed according to the ratio between the scaling factors.
The image shape is distorted and being stretched or squashed according to the scaling factors ratios.

## 4. Practical Implementation and Exploration

### 4.1 Scaling Using cv2.resize and Interpolation

In [ ]:
def compare_images(img_original, img_scaled, title_scaled):
    h1, w1 = img_original.shape[:2]
    h2, w2 = img_scaled.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [w1, w2]})

    axes[0].imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original ({w1}x{h1})")
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(img_scaled, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"{title_scaled} ({w2}x{h2})")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

scale_factor = 2.0

img_nearest = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_NEAREST)
compare_images(img, img_nearest, "Nearest Neighbor (x2.0)")

img_bilinear = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)
compare_images(img, img_bilinear, "Bilinear (x2.0)")

img_bicubic = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_CUBIC)
compare_images(img, img_bicubic, "Bicubic (x2.0)")

scale_factor = 0.5

img_nearest = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_NEAREST)
compare_images(img, img_nearest, "Nearest Neighbor (x0.5)")

img_bilinear = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)
compare_images(img, img_bilinear, "Bilinear (x0.5)")

img_bicubic = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_CUBIC)
compare_images(img, img_bicubic, "Bicubic (x0.5)")

### Interpolations
When scaling an image we change the pixels of the image and we need to fill or compress pixel values in order to keep the image similar to the original.
When scaling up there are new pixels with missing values we need to fill.
When scaling down we need to compress the image and decide which pixel values to keep and where.

### Interpolations Techniques
Nearest Neighbor - for each pixel in the scaled image look for the nearest pixel in the original image and copy it's value.
The image has high sharpness due to boundaries between pixels in the original image.
The image is not very smooth because of the sharp changes in the pixel values.

Bilinear - calculates a weighted average of the 4 closest pixels after the interpolation. The weight is calculated according to the distance to the original pixels after the interpolation.
The scaled image look more similar to the original, less sharpened because of the weighted pixel values but smoother and some of the detail are more blurry because of the averaging.

Bicubic - uses a cubic polynom over a 4x4 grid of pixels in order to fill the missing pixel values.
This performed the best at keeping the details of the original image.
The picture looks sharp and smooth.
Calculations take slightly longer, might affect real time applications.

### 4.2 Scaling Using cv2.warpAffine

In [ ]:
import numpy as np
import cv2

height, width = img.shape[:2]
scaling_factors = [0.5, 1, 2]

for sx in scaling_factors:
    for sy in scaling_factors:
        M = np.float32([[sx, 0, 0],
                        [0, sy, 0]])

        new_width = int(width * sx)
        new_height = int(height * sy)

        scaled_img = cv2.warpAffine(img, M, (new_width, new_height))
        scale_type = "Uniform" if sx == sy else "Non-Uniform"

        compare_images(img, scaled_img, f"{scale_type}: sx={sx}, sy={sy}")

### Explanations

In all the uniform scales we can see that the image kept all it proportions and geometrical properties and only scaled up or down in size.
In 0.5 the quality decreases a little bit because the image loses pixels that can improve the details.
In 2 the detail changes are similar to bilinear since warpAffine uses in its implementation.

In the non-uniform transformations the image losses proportions since the axes are not scaled in the same manner and the aspect ratio is not kept.
This also affects the image quality - for each ax if it is scaled up it will become more blurry because of the interpolation and if it is scaled down it loses details that affect image quality.